# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import os
import pandas as pd
import plotly.express as px
import numpy as np

local_path = '../data/global_energy_mix.csv'
url = 'https://raw.githubusercontent.com/dina-deifallah/data-viz-class-material-w2026/main/data/global_energy_mix.csv'
df = pd.read_csv(local_path if os.path.exists(local_path) else url)

source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

region_tot  = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_tot = df_fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
top_region, top_region_twh   = region_tot.index[0], region_tot.iloc[0]
top_country, top_country_twh = country_tot.index[0], country_tot.iloc[0]

fossil_colors = {
    'Coal':        '#D55E00',
    'Oil':         '#0072B2',
    'Natural Gas': '#56B4E9',
}

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    labels={'Source': 'Fossil source', 'TWh': 'Energy (TWh)'},
    height=700, width=1100,
    template='plotly_white'
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>'
)

fig.update_layout(
    title=(f'{top_region} burns the most fossil energy ({top_region_twh:,.0f} TWh), '
           f'but {top_country} is the single largest country ({top_country_twh:,.0f} TWh)'),
    font=dict(family='Arial', size=12),
    margin=dict(t=60, l=10, r=10, b=10),
)

fig.data[0].marker.colors = [
    c if c in fossil_colors.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
low_carbon = (df.loc[df['Source_Type'] == 'Low-carbon']
                .groupby('Country', as_index=False)['TWh'].sum()
                .sort_values('TWh', ascending=False))
low_carbon['All'] = 'Low-carbon'

leader, leader_twh = low_carbon.iloc[0]['Country'], low_carbon.iloc[0]['TWh']
runner_up, runner_twh = low_carbon.iloc[1]['Country'], low_carbon.iloc[1]['TWh']

LOW_CARBON_BLUE = '#2E75B6'
GREY = '#BBBBBB'

fig_tree = px.treemap(
    low_carbon,
    path=['All', 'Country'],
    values='TWh',
    height=600, width=1000,
    template='plotly_white'
)
fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    marker=dict(colors=['#DDDDDD' if lab == 'Low-carbon' else LOW_CARBON_BLUE
                        for lab in fig_tree.data[0].labels]),
)
fig_tree.update_layout(
    title='Low-carbon energy (nuclear + hydro) by country — treemap',
    font=dict(family='Arial', size=12),
    margin=dict(t=60, l=10, r=10, b=10),
)
fig_tree.show()

low_carbon['Highlight'] = np.where(low_carbon['Country'] == leader, 'Leader', 'Other')

fig_bar = px.bar(
    low_carbon,
    x='TWh', y='Country',
    orientation='h',
    color='Highlight',
    color_discrete_map={'Leader': LOW_CARBON_BLUE, 'Other': GREY},
    text='TWh',
    category_orders={'Country': low_carbon['Country'].tolist()},
    labels={'TWh': 'Low-carbon energy (TWh)'},
    height=600, width=1000,
    template='plotly_white'
)
fig_bar.update_traces(
    texttemplate='%{x:,.0f} TWh', textposition='outside', cliponaxis=False,
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} TWh<extra></extra>',
)
fig_bar.update_layout(
    title=(f'{leader} leads low-carbon energy with {leader_twh:,.0f} TWh — '
           f'ahead of {runner_up} ({runner_twh:,.0f} TWh)'),
    font=dict(family='Arial', size=12),
    showlegend=False,
    xaxis=dict(visible=False),
    yaxis=dict(title=''),
    bargap=0.25,
)
fig_bar.show()


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The bar chart. The bars are sorted and start from the same line, so you can see right away that France is first, then Norway and Canada. In the treemap you have to compare box sizes, and it’s hard to tell which one is bigger, especially for the small countries.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
reg_mix = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
reg_mix['Share'] = reg_mix['TWh'] / reg_mix.groupby('Region')['TWh'].transform('sum')

reg_mix = (reg_mix.set_index(['Region', 'Source'])
                  .reindex(pd.MultiIndex.from_product([reg_mix['Region'].unique(),
                                                       ['Coal', 'Oil', 'Natural Gas']],
                                                      names=['Region', 'Source']),
                           fill_value=0)
                  .reset_index())

coal = reg_mix.loc[reg_mix['Source'] == 'Coal'].sort_values('Share', ascending=False)
region_order = coal['Region'].tolist()
top_coal_region, top_coal_share = coal.iloc[0]['Region'], coal.iloc[0]['Share']

reg_mix['Label'] = np.where((reg_mix['Source'] == 'Coal') & (reg_mix['Share'] > 0),
                            reg_mix['Share'].map('{:.0%}'.format), '')

coal_accent = {
    'Coal':        fossil_colors['Coal'],
    'Oil':         '#BBBBBB',
    'Natural Gas': '#DDDDDD',
}

fig = px.bar(
    reg_mix,
    x='Share', y='Region',
    orientation='h',
    color='Source',
    text='Label',
    color_discrete_map=coal_accent,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_data={'Label': False, 'Share': ':.0%', 'TWh': ':,.0f'},
    height=550, width=1000,
    template='plotly_white'
)

fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1,
)

fig.update_layout(
    title=(f'{top_coal_region} is the most coal-dependent region — '
           f'{top_coal_share:.0%} of its fossil energy comes from coal'),
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.02, yanchor='bottom', title=''),
    margin=dict(t=100),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title=''),
)

fig.add_annotation(x=1, y=0, xref='x', yref='paper', xanchor='right', yanchor='top',
                   yshift=-4, text='100%', showarrow=False, font=dict(size=11, color='grey'))

fig.show()


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

- The treemap shows how much fossil energy each region and country uses in total, for example that Asia uses the most. The stacked bar can’t show that because every bar is 100%.
- The stacked bar shows which region depends most on coal as a share, for example Africa with 78%. In the treemap that’s hard to see because the regions are all different sizes.

*Caveat:* in this dataset Africa, Oceania and the Middle East are each represented by a single country, so "region" results there really describe South Africa, Australia and Saudi Arabia.
